# Evaluate_samples (batch-run-time fix)

Per-pipeline sample diagnostics for the fixed expiry **20110331** (10 start dates each), reconstructed **exactly** from the saved `fit_results_*.csv` artefacts — no re-optimisation, and `AugmentedLagrangian.ipynb` is not modified.

**Fix vs the first version:** the compare plot's *batch run time* now sums `run_time_sec` over **every** run in the folder (including the failed primaries of retried dates), not just the per-date winners.

In [1]:
# ── Evaluate_samples: setup ──────────────────────────────────────────────────
# Load the EXACT function definitions from AugmentedLagrangian.ipynb (generate_call_prices,
# generate_C2K, calculate_J, plot_implied_vols / blackVolatility, set_J_objective, ...) by
# executing its definition cells (0..14) into this namespace.  We never run its batch cells,
# so nothing is optimised and AugmentedLagrangian.ipynb is not modified.
import json, glob, os, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

_ALNB = json.load(open("AugmentedLagrangian.ipynb"))
for _i in range(0, 15):                       # cells 0..14 are pure definitions
    _src = "".join(_ALNB["cells"][_i]["source"])
    exec(compile(_src, f"<AL cell {_i}>", "exec"))

# match the batch's objective/normalize (meta: objective=2, normalize=True)
set_J_objective(2)
set_J_normalize(True)

OBJ = 2
N_GRID = 400          # same K-grid resolution as compare_start_dates._VC2_curve
EXPIRY = 20110331
OUT_DIR = "evaluation_samples"
os.makedirs(OUT_DIR, exist_ok=True)

# the four pipelines (fixed expiry 20110331), each one batch folder of 10 start dates
PIPELINES = [
    ("sig_nu + sig_only",        "testing_2011_2012_signu_vs_sigonly/20110331_20260627_000143"),
    ("sig_only [1]",             "testing_2011_2012_signu_vs_sigonly/20110331_20260627_024000"),
    ("sig_only [2] + sig_nu",    "testing_2011_2022_sigonly/20110331_20260628_224809"),
    ("sig_only_LKbar + sig_nu",  "testing_2011_2012_sigonlyLU/20110331_20260627_095627"),
]
print("pipelines:", [p[0] for p in PIPELINES])

pipelines: ['sig_nu + sig_only', 'sig_only [1]', 'sig_only [2] + sig_nu', 'sig_only_LKbar + sig_nu']


In [2]:
# ── parsing + reconstruction helpers ─────────────────────────────────────────
_MODE_RE = re.compile(r"(sig_nu|sig_only_LKbar|sig_only)_\d{8}_batch")

def folder_primary_mode(folder):
    """Batch's requested optimize_mode, read off the compare_summary_*.png filename
    (falls back to the mode present for the most start dates)."""
    for cs in glob.glob(os.path.join(folder, "compare_summary_*")):
        m = _MODE_RE.search(os.path.basename(cs))
        if m:
            return m.group(1)
    return None

def recon_theta(df, block):
    """Rebuild theta = cat(nus1, sigs1, nus2, sigs2) and (R1, R2) from a param_init /
    param_final block.  Verified to reproduce meta J to machine precision."""
    b = df[df["record_type"] == block]
    L = b[b["side"] == "L"].sort_values("idx")
    R = b[b["side"] == "R"].sort_values("idx")
    nus1  = torch.tensor(L["nu"].to_numpy(dtype=float))
    sigs1 = torch.tensor(L["sigma"].to_numpy(dtype=float))
    nus2  = torch.tensor(R["nu"].to_numpy(dtype=float))
    sigs2 = torch.tensor(R["sigma"].to_numpy(dtype=float))
    return torch.cat([nus1, sigs1, nus2, sigs2]), len(L), len(R)

def vc2_curve(R1, R2, S0, theta, kmin, kmax, n_grid=N_GRID):
    """EXACT copy of compare_start_dates._VC2_curve: V(K)/C''(K) on a fine K grid."""
    K  = np.linspace(kmin, kmax, n_grid)
    Kt = torch.tensor(K, dtype=torch.float64)
    with torch.no_grad():
        _, _, _, c1, c2 = calculate_J(R1, R2, S0, theta)
        c2k = generate_C2K(Kt, R1, R2, S0, theta, c1, c2).cpu().numpy()
        Ck  = generate_call_prices(Kt, R1, R2, S0, theta, c1, c2).cpu().numpy()
    V = Ck - np.maximum(S0 - K, 0.0)
    m = c2k > 0
    return np.log(K[m] / S0), np.clip(V[m] / c2k[m], 1e-30, None)

def load_pipeline(folder):
    """One WINNER record per start date (winner = max true_roughness_reduction_pct, exactly
    as compare_start_dates keeps the better of primary/retry).  Each record carries the fields
    the compare plot and the IV grid need, reconstructed from that date's fit_results CSV."""
    prim = folder_primary_mode(folder)
    Tmap = {}
    for man in glob.glob(os.path.join(folder, "manifest_*.csv")):
        mdf = pd.read_csv(man)
        Tmap.update(dict(zip(mdf["start_date"].astype(int), mdf["T"].astype(float))))
    # gather every mode-run, keyed by start date.  batch_run_time sums run_time_sec over
    # EVERY run in the folder (incl. the failed primaries of retried dates), not just the
    # per-date winners -- that is the true compute cost of the batch.
    per_date = {}
    batch_run_time = 0.0
    for f in sorted(glob.glob(os.path.join(folder, "fit_results_*.csv"))):
        df = pd.read_csv(f)
        m = df[df["record_type"] == "meta"]; meta = dict(zip(m["key"], m["value"]))
        sd = int(float(meta["dataset_start_date"]))
        true = float(meta["true_roughness_reduction_pct"])
        batch_run_time += float(meta.get("run_time_sec", 0.0) or 0.0)
        per_date.setdefault(sd, []).append((true, f, df, meta))
    recs = []
    for sd in sorted(per_date):
        true, f, df, meta = max(per_date[sd], key=lambda t: t[0])   # winner
        S0 = float(meta["S0"])
        mk = df[df["record_type"] == "market"].copy()
        strikes = mk["strike"].to_numpy(dtype=float)
        kmin, kmax = float(strikes.min()), float(strikes.max())
        th_i, R1i, R2i = recon_theta(df, "param_init")
        th_f, R1f, R2f = recon_theta(df, "param_final")
        x_i, vc2_i = vc2_curve(R1i, R2i, S0, th_i, kmin, kmax)
        x_f, vc2_f = vc2_curve(R1f, R2f, S0, th_f, kmin, kmax)
        recs.append(dict(
            start_date=sd, mode=str(meta["optimize_mode"]), S0=S0,
            T=float(Tmap.get(sd, np.nan)),
            J_init=float(meta["J_initial"]), J_final=float(meta["J_final"]),
            pct_improve=float(meta["J_pct_improvement"]),
            true_reduce=true, S_final=float(meta["S_final"]),
            sqrt_S_final=float(meta["sqrt_S_final"]),
            run_time=float(meta.get("run_time_sec", np.nan)),
            R1_init=R1i, R2_init=R2i, R1_final=R1f, R2_final=R2f,
            x_init=x_i, vc2_init=vc2_i, x_final=x_f, vc2_final=vc2_f,
            strikes=strikes, bid=mk["bid"].to_numpy(dtype=float),
            ask=mk["ask"].to_numpy(dtype=float),
            C_initial=mk["C_initial"].to_numpy(dtype=float),
            C_final=mk["C_final"].to_numpy(dtype=float),
        ))
    return prim, recs, batch_run_time

In [3]:
# ── figure 1: trimmed compare plot (V/C'' before, V/C'' after, true reduction %) ──
# Panels (b), (c) and (T) of compare_start_dates._emit_batch_summary reproduced verbatim,
# with the same suptitle, italic run-info line and grey stats box.
def plot_compare3(name, primary_mode, recs, outpath, batch_run_time):
    labels = [str(r["start_date"]) for r in recs]
    cmap = plt.cm.viridis(np.linspace(0, 0.9, len(recs)))
    optimize_mode = primary_mode
    modes_pd = [r["mode"] for r in recs]
    fb_idx = [k for k, m in enumerate(modes_pd) if m != optimize_mode]
    fb_modes = sorted(set(modes_pd[k] for k in fb_idx))
    def _mark_fallback(bars):
        for k in fb_idx:
            bars[k].set_hatch("//"); bars[k].set_edgecolor("red"); bars[k].set_linewidth(1.3)

    fig, (ax1, ax2, axT) = plt.subplots(1, 3, figsize=(24, 7))

    # (b) V/C'' before
    for r, c in zip(recs, cmap):
        ax1.semilogy(r["x_init"], r["vc2_init"], "-", color=c, lw=1.2, label=str(r["start_date"]))
    ax1.axvline(0.0, color="gray", ls="--", lw=0.8)
    ax1.set_xlabel("log(K / S0)"); ax1.set_ylabel("V / C''  (log)")
    ax1.set_title("V/C'' BEFORE optimization")
    ax1.grid(alpha=0.3, which="both"); ax1.legend(fontsize=6, ncol=2)

    # (c) V/C'' after
    for r, c in zip(recs, cmap):
        ax2.semilogy(r["x_final"], r["vc2_final"], "-", color=c, lw=1.2, label=str(r["start_date"]))
    ax2.axvline(0.0, color="gray", ls="--", lw=0.8)
    ax2.set_xlabel("log(K / S0)"); ax2.set_ylabel("V / C''  (log)")
    ax2.set_title(f"V/C'' AFTER optimization (mode={optimize_mode})")
    ax2.grid(alpha=0.3, which="both"); ax2.legend(fontsize=6, ncol=2)

    # (T) TRUE roughness reduction % per start date
    _mark_fallback(axT.bar(labels, [r["true_reduce"] for r in recs], color=cmap))
    axT.set_ylabel("TRUE roughness reduction (%)"); axT.set_xlabel("start date")
    axT.set_title(f"TRUE % improvement (batch01)")
    axT.grid(alpha=0.3, axis="y")
    plt.setp(axT.get_xticklabels(), rotation=90, fontsize=7)
    for i, r in enumerate(recs):
        axT.text(i, r["true_reduce"], f"{r['true_reduce']:.1f}", ha="center", va="bottom", fontsize=6)

    # suptitle + italic run-info line (same wording as _emit_batch_summary)
    fig.suptitle(f"Single-mode summary | fixed expiry {EXPIRY} | batch01 "
                 f"({labels[0]}..{labels[-1]})", fontsize=14, fontweight="bold", y=0.99)
    # batch run time = sum over EVERY run in the folder (incl. failed primaries of retries)
    _rt = float(batch_run_time)
    _mx1 = recs[0]["R1_final"] // max(1, recs[0]["R1_init"])
    _mx2 = recs[0]["R2_final"] // max(1, recs[0]["R2_init"])
    fig.text(0.5, 0.94,
             f"Mode:  {optimize_mode}   |   obj={OBJ}   |   mult={_mx1}x{_mx2}   |   "
             f"batch run time: {_rt:.1f}s ({_rt/60.0:.1f} min)"
             + (f"   |   fallback: {len(fb_idx)} date(s) -> {','.join(fb_modes)} "
                f"(red-hatched bars)" if fb_idx else ""),
             ha="center", fontsize=14, style="italic", color="#333333")

    # grey stats box (identical fields/format to _emit_batch_summary)
    pcts = [r["pct_improve"] for r in recs]; trs = [r["true_reduce"] for r in recs]
    stats_text = (
        f"log-J improvement (%) — "
        f"Mean: {np.mean(pcts):6.2f}%  |  Median: {np.median(pcts):6.2f}%  |  "
        f"Min: {np.min(pcts):6.2f}%  |  Max: {np.max(pcts):6.2f}%  |  Std: {np.std(pcts):6.2f}%\n"
        f"TRUE roughness reduction (%) — "
        f"Mean: {np.mean(trs):6.2f}%  |  Median: {np.median(trs):6.2f}%  |  "
        f"Min: {np.min(trs):6.2f}%  |  Max: {np.max(trs):6.2f}%  |  Std: {np.std(trs):6.2f}%\n"
        f"ABSOLUTE initial roughness — "
        f"S_init median: {np.median([np.exp(r['J_init']) for r in recs]):.4g}  |  "
        f"sqrt(S_init) median: {np.median([np.exp(r['J_init']/2.0) for r in recs]):.4g}\n"
        f"ABSOLUTE final roughness — "
        f"S_final median: {np.median([r['S_final'] for r in recs]):.4g}  |  "
        f"sqrt(S_final) median: {np.median([r['sqrt_S_final'] for r in recs]):.4g}")
    fig.text(0.5, 0.005, stats_text, ha="center", fontsize=13,
             bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8))

    fig.tight_layout(rect=[0, 0.10, 1, 0.90])
    fig.savefig(outpath, dpi=140, bbox_inches="tight")
    fig.savefig(os.path.splitext(outpath)[0] + ".svg", bbox_inches="tight")
    plt.close(fig)
    print(f"[compare3] {name} -> {outpath}")

In [4]:
# ── figure 2: 10 (cols) x 2 (rows) implied-vol grid ──────────────────────────
# Reuses AugmentedLagrangian.ipynb's exact IV machinery (blackVolatility, the _good filter,
# the median-bid/ask-spread "clean window", and the per-panel drawing) so each column looks
# identical to that date's implied_volatility panel.  Top row = full smile, bottom = zoomed.
def _iv_panels_for_date(r):
    """Return (x, iv_init, iv_final, iv_bid, iv_ask, clean_keep) exactly like plot_implied_vols."""
    strikes_np = r["strikes"]; S0 = r["S0"]; T = r["T"]
    C_init_np = r["C_initial"]; C_final_np = r["C_final"]
    bids_np = r["bid"]; asks_np = r["ask"]
    iv_floor, iv_cap = 1e-3, 3.0
    x = np.log(strikes_np / S0)
    iv_init  = np.array([blackVolatility(1.0, S0, float(k), T, float(C_init_np[i]))  for i, k in enumerate(strikes_np)])
    iv_final = np.array([blackVolatility(1.0, S0, float(k), T, float(C_final_np[i])) for i, k in enumerate(strikes_np)])
    iv_bid = np.array([blackVolatility(1.0, S0, float(k), T, float(bids_np[i])) for i, k in enumerate(strikes_np)])
    iv_ask = np.array([blackVolatility(1.0, S0, float(k), T, float(asks_np[i])) for i, k in enumerate(strikes_np)])
    def _good(v):
        return np.isfinite(v) & (v > iv_floor) & (v < iv_cap)
    both = _good(iv_bid) & _good(iv_ask)
    clean_keep = np.zeros_like(x, dtype=bool)
    if both.sum() >= 4:
        spr = (iv_ask - iv_bid)[both]
        tight = both.copy(); tight[both] = spr <= np.median(spr)
        if tight.any():
            xlo, xhi = x[tight].min(), x[tight].max()
            clean_keep = (x >= xlo) & (x <= xhi)
    if not clean_keep.any():
        clean_keep = both.copy() if both.any() else np.ones_like(x, dtype=bool)
    return x, iv_init, iv_final, iv_bid, iv_ask, clean_keep, _good

def _draw_iv_panel(ax, r, keep, title, show_legend=False):
    x, iv_init, iv_final, iv_bid, iv_ask, _, _good = r["_iv"]
    ma = _good(iv_ask) & keep; mb = _good(iv_bid) & keep
    ax.scatter(x[ma], iv_ask[ma], s=16, color="firebrick", marker="^", label="Market ask IV")
    ax.scatter(x[mb], iv_bid[mb], s=16, color="green",     marker="v", label="Market bid IV")
    vi = _good(iv_init) & keep; vf = _good(iv_final) & keep
    ax.plot(x[vi], iv_init[vi], "-", color="steelblue", lw=1.2, label="Initial IV (arb-free)")
    ax.plot(x[vf], iv_final[vf], "-", color="tomato",   lw=1.6, label="Final IV (LVG fit)")
    ax.axvline(0.0, color="gray", ls="--", lw=0.8, label="ATM (K=S0)")
    xs = [x[ma], x[mb], x[vi], x[vf]]; ys = [iv_ask[ma], iv_bid[mb], iv_init[vi], iv_final[vf]]
    ax_x = np.concatenate([a for a in xs if a.size]) if any(a.size for a in xs) else np.array([])
    ax_y = np.concatenate([a for a in ys if a.size]) if any(a.size for a in ys) else np.array([])
    if ax_x.size:
        xpad = 0.03 * (ax_x.max() - ax_x.min() + 1e-9); ypad = 0.08 * (ax_y.max() - ax_y.min() + 1e-9)
        ax.set_xlim(ax_x.min() - xpad, ax_x.max() + xpad)
        ax.set_ylim(max(0.0, ax_y.min() - ypad), ax_y.max() + ypad)
    ax.set_xlabel("log(K / S0)", fontsize=8); ax.set_ylabel("Implied Vol", fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.tick_params(labelsize=7); ax.grid(alpha=0.3)
    if show_legend:
        ax.legend(fontsize=6)

def plot_iv_grid(name, recs, outpath):
    n = len(recs)
    for r in recs:
        r["_iv"] = _iv_panels_for_date(r)
    fig, axes = plt.subplots(2, n, figsize=(4.2 * n, 9), squeeze=False)
    for j, r in enumerate(recs):
        _, _, _, _, _, clean_keep, _ = r["_iv"]
        full = np.ones(len(r["strikes"]), dtype=bool)
        _draw_iv_panel(axes[0][j], r, full, f"{r['start_date']}  (T={r['T']:.3g})", show_legend=(j == 0))
        _draw_iv_panel(axes[1][j], r, clean_keep, f"{r['start_date']}  zoomed", show_legend=(j == 0))
    axes[0][0].annotate("Implied Volatility Smile", xy=(0, 0.5), xytext=(-axes[0][0].yaxis.labelpad - 30, 0),
                        xycoords=axes[0][0].yaxis.label, textcoords="offset points",
                        fontsize=12, fontweight="bold", ha="right", va="center", rotation=90)
    axes[1][0].annotate("Zoomed (near-money)", xy=(0, 0.5), xytext=(-axes[1][0].yaxis.labelpad - 30, 0),
                        xycoords=axes[1][0].yaxis.label, textcoords="offset points",
                        fontsize=12, fontweight="bold", ha="right", va="center", rotation=90)
    fig.suptitle(f"Implied Volatility smiles — {name} | fixed expiry {EXPIRY} | {n} start dates",
                 fontsize=15, fontweight="bold")
    fig.tight_layout(rect=[0.015, 0, 1, 0.96])
    fig.savefig(outpath, dpi=140, bbox_inches="tight")
    fig.savefig(os.path.splitext(outpath)[0] + ".svg", bbox_inches="tight")
    plt.close(fig)
    print(f"[iv_grid] {name} -> {outpath} ({n} dates)")

In [5]:
# ── run: produce both figures for every pipeline ─────────────────────────────
def _slug(name):
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()

for name, folder in PIPELINES:
    if not os.path.isdir(folder):
        print(f"[skip] {name}: folder not found -> {folder}")
        continue
    primary_mode, recs, batch_run_time = load_pipeline(folder)
    winners_only = float(sum(r.get("run_time", 0.0) for r in recs))
    print(f"\n=== {name}  ({folder})  primary={primary_mode}  dates={len(recs)} ===")
    n_runs = len(glob.glob(os.path.join(folder, "fit_results_*.csv")))
    print(f"    batch run time: ALL runs={batch_run_time:.1f}s  |  winners-only={winners_only:.1f}s  "
          f"(diff={batch_run_time - winners_only:.1f}s; {n_runs} run files over {len(recs)} dates)")
    slug = _slug(name)
    plot_compare3(name, primary_mode, recs, os.path.join(OUT_DIR, f"compare3_{slug}.png"), batch_run_time)
    plot_iv_grid(name, recs, os.path.join(OUT_DIR, f"iv_grid_{slug}.png"))

print("\nAll pipelines done. Images in", OUT_DIR)


=== sig_nu + sig_only  (testing_2011_2012_signu_vs_sigonly/20110331_20260627_000143)  primary=sig_nu  dates=10 ===
    batch run time: ALL runs=94.1s  |  winners-only=85.4s  (diff=8.7s; 11 run files over 10 dates)
[compare3] sig_nu + sig_only -> evaluation_samples/compare3_sig_nu_sig_only.png
[iv_grid] sig_nu + sig_only -> evaluation_samples/iv_grid_sig_nu_sig_only.png (10 dates)

=== sig_only [1]  (testing_2011_2012_signu_vs_sigonly/20110331_20260627_024000)  primary=sig_only  dates=10 ===
    batch run time: ALL runs=86.1s  |  winners-only=86.1s  (diff=0.0s; 10 run files over 10 dates)
[compare3] sig_only [1] -> evaluation_samples/compare3_sig_only_1.png
[iv_grid] sig_only [1] -> evaluation_samples/iv_grid_sig_only_1.png (10 dates)

=== sig_only [2] + sig_nu  (testing_2011_2022_sigonly/20110331_20260628_224809)  primary=sig_only  dates=10 ===
    batch run time: ALL runs=88.7s  |  winners-only=88.7s  (diff=0.0s; 10 run files over 10 dates)
[compare3] sig_only [2] + sig_nu -> evaluat